# Module 3 · Photosynthetic Saturation and Canopy Design
**20 minutes: principles 4 · worked example 4 · transfer exercises 9 · discussion 3**

[Course introduction](00_Introduction.ipynb) · [Runnable reference answer](#reference-answer)

**Learning objectives:** implement net A-Q and area weighting; run a controlled angle trial;
explain why photon interception and CO2 uptake need not rank designs in the same order.


In [ ]:
# Run this preparation cell once after starting/restarting the kernel.
# Finds normal, flattened, nested and ZIP-preserved course uploads.
from pathlib import Path
import sys,os,subprocess,hashlib,zipfile,shutil

def locate_canopy_runtime(input_root=None,local_roots=None,work=None,tag=None):
    tag=tag or ('win_amd64' if sys.platform=='win32' else 'manylinux_2_34_x86_64')
    expected=f'canopyteach-1.0.1-cp311-abi3-{tag}.whl'
    pattern=f'canopyteach-1.0.1-cp311-abi3-{tag}*.whl'
    work=Path(work or '/kaggle/working/canopyteach_practical_en')
    candidate_root=Path(input_root or '/kaggle/input')
    wheels=set();helpers=set();archives=set();observed=[];archive_errors=[]

    def consider(p):
        if p.name=='canopy_practical.py':helpers.add(p.resolve())
        elif p.match(pattern):wheels.add(p.resolve())
        elif p.suffix.lower()=='.zip':archives.add(p.resolve())
        if len(observed)<35 and p.is_file():observed.append(str(p))

    if candidate_root.is_dir():
        # Kaggle can add dataset/owner/version directories or flatten uploads.
        for p in candidate_root.rglob('*'):
            if p.is_file():consider(p)
    else:
        roots=local_roots if local_roots is not None else ([] if input_root is not None else [Path.cwd(),*Path.cwd().parents])
        for folder in map(Path,roots):
            if folder==Path(folder.anchor):continue
            for p in [folder/'canopy_practical.py',*folder.glob(pattern),*folder.glob('wheels/'+pattern)]:
                if p.is_file():consider(p)
        if input_root is not None:
            observed.append('Input directory does not exist: '+str(candidate_root))

    # Read only the runtime members of course ZIPs into a writable directory.
    # Never extract arbitrary archive paths or change the input Dataset.
    if not wheels or not helpers:
        for archive in sorted(archives):
            try:
                with zipfile.ZipFile(archive) as z:
                    selected=[n for n in z.namelist() if not n.endswith('/') and
                        (Path(n).name=='canopy_practical.py' or Path(n).match(pattern))]
                    if not selected:continue
                    key=hashlib.sha256(str(archive).encode()).hexdigest()[:12]
                    for i,name in enumerate(selected):
                        info=z.getinfo(name)
                        if info.file_size>30_000_000:raise ValueError('Runtime member exceeds 30 MB: '+name)
                        dest=work/'_runtime'/key/str(i)/Path(name).name
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        dest.write_bytes(z.read(name));consider(dest)
            except (zipfile.BadZipFile,OSError,ValueError) as exc:
                archive_errors.append(str(archive)+': '+str(exc))

    if not wheels or not helpers:
        missing=[]
        if not wheels:missing.append(expected)
        if not helpers:missing.append('canopy_practical.py')
        details='\n'.join(observed) or '(no mounted input files found)'
        raise FileNotFoundError(
            'Missing course runtime: '+', '.join(missing)+'\n'
            'Add the course Dataset as an input to THIS notebook; importing an .ipynb alone does not attach its Dataset.\n'
            'Searched input: '+str(candidate_root)+'\nObserved files (first 35):\n'+details+
            ('\nArchive errors:\n'+'\n'.join(archive_errors) if archive_errors else ''))

    # Prefer a nearby helper, without requiring the wheels/ directory layout.
    pairs=[]
    for wheel in sorted(wheels):
        for helper in sorted(helpers):
            shared=len(os.path.commonpath([str(wheel.parent),str(helper.parent)]).split(os.sep))
            nearby=helper.parent in (wheel.parent,wheel.parent.parent)
            pairs.append((not nearby,-shared,str(wheel),str(helper)))
    _,_,wheel_name,helper_name=min(pairs)
    wheel,helper=Path(wheel_name),Path(helper_name)
    # Browser downloads may add a suffix such as " (1)" to a wheel filename.
    if wheel.name!=expected:
        normalized=work/'_runtime'/'normalized'/expected
        normalized.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(wheel,normalized);wheel=normalized
    return wheel,helper

if __name__=='__main__':
    if sys.version_info<(3,11):raise RuntimeError('This course requires CPython 3.11 or newer.')
    input_override=os.environ.get('CANOPYTEACH_INPUT_ROOT')
    work_override=os.environ.get('CANOPYTEACH_WORK_ROOT')
    on_kaggle=Path('/kaggle/working').is_dir()
    WORK=Path(work_override) if work_override else (
        Path('/kaggle/working/canopyteach_practical_en') if on_kaggle else Path.cwd()/'results/practical_en')
    wheel,helper=locate_canopy_runtime(input_root=input_override,work=WORK)
    PACKAGE_ROOT=helper.parent
    WORK.mkdir(parents=True,exist_ok=True);site=WORK/'_packages';marker=site/'course_wheel.sha256'
    signature=hashlib.sha256(wheel.read_bytes()).hexdigest()
    if not marker.exists() or marker.read_text().strip()!=signature:
        if 'canopyteach' in sys.modules:
            raise RuntimeError('Restart the kernel before installing a different course wheel, then run preparation again.')
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--disable-pip-version-check',
                        '--upgrade','--target',str(site),str(wheel)],check=True)
        marker.write_text(signature)
    sys.path.insert(0,str(site));sys.path.insert(0,str(PACKAGE_ROOT))
    import canopy_practical as cp
    import canopyteach
    if canopyteach.__version__!='1.0.1':raise RuntimeError('Unexpected canopyteach version; restart the kernel.')
    cp.style()
    try:get_ipython().run_line_magic('matplotlib','inline')
    except NameError:pass
    DEFAULT_STEP=int(os.environ.get('CANOPYTEACH_STEP_MINUTES','30'))
    DEFAULT_SPACING=float(os.environ.get('CANOPYTEACH_RAY_SPACING','.005'))
    print('Runtime wheel:',wheel)
    print('Teaching helper:',helper)
    print('Ready. Outputs:',WORK)
    print('Random whole-plant azimuth; central crop; XY periodic; C/OpenMP; synthetic direct light.')


## 1. Principles · 4 min

$$A=\frac{\alpha Q+P_{max}-\sqrt{(\alpha Q+P_{max})^2-4\theta\alpha QP_{max}}}{2\theta}-R_d.$$
Use alpha=.05, Pmax=30, theta=.7 and Rd=1. Thus **A(0)=-1**.
This is a teaching A-Q response, not a biochemical C4 model.

Canopy rate is $A_c=\sum_i A_i A_{leaf,i}/A_g$.
Evaluate A on each fragment before area-weighting. Do not average by face count or apply
nonlinear A to mean PPFD. Daily leaf net uptake includes nighttime leaf respiration;
stem and root respiration are outside this indicator.


## 2. Worked example · 4 min

Run the supplied `assimilation()` and inspect the response curve.
For two leaves of **1 m2 each**, PPFD treatments 1600+0 and 800+800 have equal total
photons, but net uptake is approximately **24.17965** and **41.93881 umol CO2/s**.

The next cell asks you to implement the equation and aggregate leaves of unequal area.


In [ ]:
import matplotlib.pyplot as plt
from canopyteach import assimilation
import canopy_practical as cp
cp.style()

print('A(0), A(800), A(1600):',assimilation([0,800,1600]))
print('Two 1m2 leaves, 1600+0:',float(assimilation([1600,0]).sum()),'umol CO2/s')
print('Two 1m2 leaves, 800+800:',float(assimilation([800,800]).sum()),'umol CO2/s')
fig=cp.aq_example();plt.show()


## 3. Transfer programming exercises 3A–3B · 5 min

**3A · Implement net A-Q.** Complete the return expression in `student_leaf_A` using
the equation above. Array conversion and intermediate values are supplied.
Retain `-rd` and support arrays. For Q=[0,800,1600], expect approximately
**[-1,20.969404,25.179655]**.

**3B · Weight by actual area.** Complete the aggregation expression.
Two leaf fragments have areas **0.5 and 1.5 m2**, PPFD **1600 and 0**, and ground area **1 m2**.
The canopy rate should be **11.0898274 umol CO2/(m2 ground s)**.
The fragments differ in area, so a mean by fragment count is incorrect.


In [ ]:
import numpy as np
from canopyteach import assimilation

def student_leaf_A(q,alpha=.05,pmax=30.,theta=.7,rd=1.):
    q=np.asarray(q,dtype=float)
    x=alpha*q
    discriminant=(x+pmax)**2-4*theta*x*pmax
    return None  # TODO 3A: translate the A-Q equation; do not remove dark respiration.

def student_canopy_rate(leaf_A,leaf_area,ground_area):
    return None  # TODO 3B: area-weight the leaf rates, then divide by ground area.

probe=student_leaf_A(np.array([0.,800.,1600.]))
if probe is None:
    print('3A is unfinished: complete return; A(0) should be -1, not zero.')
else:
    np.testing.assert_allclose(probe,assimilation([0,800,1600]),rtol=1e-8,atol=1e-8)
    print('3A self-check passed:',probe)
    leaf_A=student_leaf_A([1600,0])
    value=student_canopy_rate(leaf_A,np.array([.5,1.5]),1.)
    if value is None:
        print('3B is unfinished: these leaf areas differ, so do not average by face count.')
    else:
        np.testing.assert_allclose(value,11.089827398725358)
        print('3B canopy rate:',value,'umol CO2 / m2 ground / s')


## 4. Independent design experiment 3C · 4 min

Change only `my_lower_angle` and `my_upper_angle`, measured above horizontal, then rerun
the whole cell. Even if 3A is unfinished, the supplied equation produces the treatments,
table and figure. After your equation passes its self-check, the trial uses it.
The output explicitly identifies the active equation.

Complete **both** relative-change expressions: `100*(candidate/baseline - 1)` for daily
net CO2 and for daily leaf photons. Record the angles and LAI; write two sentences explaining
the result and its scope. A decrease is a valid outcome. Do not also alter density or physiology.

With the supplied 10/55-degree trial, daily CO2 changes by about **-1.983%** while daily
leaf photons change by about **+0.961%**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import canopy_practical as cp
cp.style()

my_lower_angle=10.  # EDIT: degrees above horizontal; change angles only.
my_upper_angle=55.  # EDIT: degrees above horizontal.
equation=globals().get('student_leaf_A')
design_results,equation_label=cp.design_trial(my_lower_angle,my_upper_angle,rate_function=equation,
    step_minutes=DEFAULT_STEP,ray_spacing=DEFAULT_SPACING)
print('Active leaf equation:',equation_label)
cp.print_design(design_results)
baseline=design_results['baseline'].summary
candidate=design_results['my_design'].summary
carbon_change_percent=None  # TODO 3C: 100*(candidate daily CO2 / baseline daily CO2 - 1)
photon_change_percent=None  # TODO 3C: same relative-change formula for daily leaf photons.
if carbon_change_percent is None or photon_change_percent is None:
    print('The table and figure are ready. Complete both contrasts, then explain whether their signs agree.')
else:
    np.testing.assert_allclose(carbon_change_percent,100*(candidate['daily_net_mol_co2_m2_ground']/baseline['daily_net_mol_co2_m2_ground']-1))
    np.testing.assert_allclose(photon_change_percent,100*(candidate['daily_leaf_intercepted_mol_photons_m2_ground']/baseline['daily_leaf_intercepted_mol_photons_m2_ground']-1))
    print('Changes (%): CO2',carbon_change_percent,'photons',photon_change_percent)
fig=cp.daily_figure(design_results);plt.show()
print('Saved:',cp.save_experiment(design_results,WORK/'module3_design'))


<a id="reference-answer"></a>
## 5. Reference answers and discussion · 3 min

**3A:** retain the full net A-Q equation and dark respiration.
**3B:** `sum(A*leaf_area)/ground_area`.
**3C:** divide by the baseline; hold date, weather, actual LAI, density, physiology and sampling fixed.

In the example, slightly more photons produce less daily net CO2, demonstrating the
importance of spatial and temporal allocation. These totals alone do not prove that every
layer is more uniform at every time. The result is not a universal or field-optimal ideotype.

The local reference below respects your selected angles, or uses 10/55 in a fresh kernel.
It preserves your function and parameter choices. Submit the equations, two contrasts,
results table and one sentence stating the conditions to which your conclusion applies.


In [ ]:
def _reference_design(lower,upper,step,spacing):
    import numpy as np
    import matplotlib.pyplot as plt
    import canopy_practical as cp
    def net_a(q,alpha=.05,pmax=30.,theta=.7,rd=1.):
        q=np.asarray(q,dtype=float);x=alpha*q
        return (x+pmax-np.sqrt((x+pmax)**2-4*theta*x*pmax))/(2*theta)-rd
    def canopy_rate(A,area,ground):return np.sum(np.asarray(A)*np.asarray(area))/ground
    print('Unequal-area example:',canopy_rate(net_a([1600,0]),[.5,1.5],1.))
    results,label=cp.design_trial(lower,upper,rate_function=net_a,step_minutes=step,ray_spacing=spacing)
    cp.print_design(results)
    b=results['baseline'].summary;c=results['my_design'].summary
    dc=100*(c['daily_net_mol_co2_m2_ground']/b['daily_net_mol_co2_m2_ground']-1)
    dl=100*(c['daily_leaf_intercepted_mol_photons_m2_ground']/b['daily_leaf_intercepted_mol_photons_m2_ground']-1)
    print('CO2 change (%):',dc,'leaf-photon change (%):',dl)
    fig=cp.daily_figure(results);plt.show()
_reference_design(globals().get('my_lower_angle',10.),globals().get('my_upper_angle',55.),DEFAULT_STEP,DEFAULT_SPACING)
